# Suspicious Login Attack Detector

This notebook implements a complete suspicious-login detection prototype:

1. Generate **3,000 synthetic rejected-login records** using 10 common email accounts.
2. Create chronological behavior features without future-data leakage.
3. Train a logistic-regression attack classifier.
4. Combine model probability with deterministic security rules.
5. Add a `thret` score from `0` to `1` and recommend an action.
6. Evaluate any new rejected-login attempt by calling one method.

> **Production warning:** The included labels and patterns are synthetic. Retrain and calibrate the model with confirmed real security outcomes before production use.


## 1. Environment

Required packages: `pandas`, `numpy`, `scikit-learn`, `joblib`, and `matplotlib`.

Uncomment the installation command only when the packages are missing.


## 2. Imports, schema, features, and thresholds

In [2]:
from __future__ import annotations
from sklearn.base import BaseEstimator, TransformerMixin

import inspect
import json
import math
import random
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


## 3. Synthetic data generation

The generator creates normal user mistakes and four attack scenarios:

- Brute force against one account
- Credential stuffing across multiple accounts
- Distributed attacks against one account
- Possible account takeover from a new device, IP, or location

Documentation-only IP ranges are used, so the generated addresses do not represent real customers.


In [4]:
df =pd.read_csv(Path('../data/login_reject_history_3000.csv'))
df['time_to_attempt']=pd.to_datetime(df['time_to_attempt'])


Index(['id', 'email', 'device_mac_id', 'ip', 'location', 'time_to_attempt',
       'rejection_reason', 'is_suspicious', 'scenario'],
      dtype='object')

In [4]:
 


class ApplyTimeChangeTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        sort_by,
        identity,
        target,
        output_feature,
        condition
    ):
        '''
        sort_by ='time_to_attempt'
        output_feature ='email_attempts_5m'
        identity ='email'
        target ="time_to_attempt"
        condition =lambda seconds: seconds.gt(5 * 60) 
        '''
        self.sort_by = sort_by
        self.identity = identity
        self.target = target
        self.output_feature = output_feature
        self.condition =condition

        
    def fit(self, X, y=None):
        return self

    def transform(self, X):

        # Always use X provided by ColumnTransformer
        tmp_df = X.copy()

        # Validate required columns
        required_columns = {
            self.sort_by,
            self.identity,
            self.target
        }

        missing_columns = required_columns - set(tmp_df.columns)

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {sorted(missing_columns)}"
            )

        # Remember original row positions
        original_position_column = "_original_position"

        tmp_df[original_position_column] = np.arange(
            len(tmp_df)
        )

        # Ensure timestamp is datetime
        tmp_df[self.target] = pd.to_datetime(
            tmp_df[self.target],
            errors="raise"
        )

        # Sort newest to oldest
        tmp_df = tmp_df.sort_values(
            by=self.sort_by,
            ascending=False,
            kind="stable"
        )

        # Calculate difference from next older attempt
        seconds_diff = (
            tmp_df
            .groupby(
                self.identity,
                sort=False
            )[self.target]
            .diff(periods=-1)
            .dt.total_seconds()
        )

        condition_result = self.condition(seconds_diff)

        # More than time limit -> 1
        # Equal/below limit or no older attempt -> 0
        tmp_df[self.output_feature] = np.where(
            condition_result,
            1,
            0
        )

        # Restore original input order
        tmp_df = tmp_df.sort_values(
            by=original_position_column,
            kind="stable"
        )

        # Return exactly one output column
        result = tmp_df[
            [self.output_feature]
        ].copy()

        # Ensure index matches original X
        result.index = X.index

        return result
    def get_feature_names_out(self, input_features=None):
        return np.array([self.output_feature])

# Missing-Value Rules for 22 Engineered Features

| Feature                              | Missing-value rule                                                    |
| ------------------------------------ | --------------------------------------------------------------------- |
| `seconds_since_email_last_attempt`   | Set to `2592000` seconds — 30 days                                    |
| `email_attempts_5m`                  | `0`                                                                   |
| `email_attempts_1h`                  | `0`                                                                   |
| `email_attempts_24h`                 | `0`                                                                   |
| `consecutive_email_failures`         | `0`                                                                   |
| `seconds_since_ip_last_attempt`      | Set to `2592000`                                                      |
| `ip_attempts_5m`                     | `0`                                                                   |
| `ip_attempts_1h`                     | `0`                                                                   |
| `ip_unique_emails_10m`               | `0`                                                                   |
| `ip_unique_emails_1h`                | `0`                                                                   |
| `device_unique_emails_1h`            | `0`                                                                   |
| `email_unique_ips_1h`                | `0`                                                                   |
| `email_unique_ips_24h`               | `0`                                                                   |
| `email_unique_devices_24h`           | `0`                                                                   |
| `email_unique_locations_24h`         | `0`                                                                   |
| `is_new_ip_for_email`                | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_device_for_email`            | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_location_for_email`          | `0` when the email has no previous history; otherwise calculate `0/1` |
| `location_changed_from_last_attempt` | `0` when no previous attempt exists                                   |
| `seconds_since_device_last_attempt`  | Set to `2592000`                                                      |
| `hour_sin`                           | Recalculate from `time_to_attempt`; do not statistically impute       |
| `hour_cos`                           | Recalculate from `time_to_attempt`; do not statistically impute       |




In [5]:
df

,id,email,device_mac_id,ip,location,time_to_attempt,rejection_reason,is_suspicious,scenario
0,1,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01 00:11:51+00:00,expired_session,0,normal_user_error
1,2,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01 00:28:22+00:00,invalid_password,0,normal_user_error
2,3,user1@example.com,02:39:0C:8C:7D:72,192.0.2.36,"Dhaka, BD",2026-07-01 00:37:27+00:00,password_typo,0,normal_user_error
3,4,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01 00:39:58+00:00,password_typo,0,normal_user_error
4,5,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-01 01:50:21+00:00,expired_session,0,normal_user_error
...,...,...,...,...,...,...,...,...,...
2995,2996,user10@example.com,02:A1:4D:E1:22:F0,192.0.2.91,"New York, US",2026-07-31 20:10:34+00:00,invalid_password,0,normal_user_error
2996,2997,user2@example.com,02:34:2C:D8:10:0F,192.0.2.24,"Chattogram, BD",2026-07-31 20:17:57+00:00,otp_failed,0,normal_user_error
2997,2998,user10@example.com,02:8E:E8:BA:53:BD,192.0.2.91,"Chattogram, BD",2026-07-31 22:47:08+00:00,password_typo,0,normal_user_error
2998,2999,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-31 22:49:41+00:00,password_typo,0,normal_user_error


# Zero Imputer

In [6]:
#  # 
# zero_default_features = [
#    "email_attempts_5m",
#     "email_attempts_1h",
#     "email_attempts_24h",
#     "consecutive_email_failures",
#     "ip_attempts_5m",
#     "ip_attempts_1h",
#     "ip_unique_emails_10m",
#     "ip_unique_emails_1h",
#     "device_unique_emails_1h",
#     "email_unique_ips_1h",
#     "email_unique_ips_24h",
#     "email_unique_devices_24h",
#     "email_unique_locations_24h",
#     "is_new_ip_for_email",
#     "is_new_device_for_email",
#     "is_new_location_for_email",
#     "location_changed_from_last_attempt"
# ]





# df[zero_default_features]=np.nan
# df[['seconds_since_email_last_attempt',
#    'seconds_since_device_last_attempt',
#    'is_new_location_for_email',
#    'is_new_device_for_email',
#    'is_new_ip_for_email',
#    'location_changed_from_last_attempt']]=np.nan 

# preprocessor =ColumnTransformer(
#     transformers=[
#         (
#             'email_attempts_5m',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='email_attempts_5m',
#                 identity='email',
#                 condition=lambda second: second.gt(5*60)),
#               ["email", "time_to_attempt"]
#          ),
#         (
#             'email_attempts_1h',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='email_attempts_1h',
#                 identity='email',
#                 condition=lambda second: second.gt(1*60*60)),
#               ["email", "time_to_attempt"]
#          ),
#         (
#             'email_attempts_24h',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='email_attempts_24h',
#                 identity='email',
#                 condition=lambda second: second.gt(24*60*60)),
#               ["email", "time_to_attempt"]
#          ),
#         (
#             'ip_attempts_5m',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='ip_attempts_5m',
#                 identity='ip',
#                 condition=lambda second: second.gt(5*60)),
#               ["ip", "time_to_attempt"]
#          ),
#         (
#             'ip_attempts_1h',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='ip_attempts_1h',
#                 identity='ip',
#                 condition=lambda second: second.gt(1*60*60)),
#               ["ip", "time_to_attempt"]
#          ),
#
#         (
#             "keep_email_and_time",
#             "passthrough",
#             ["email", "time_to_attempt",'ip']
#         )
#     ],
#     remainder='passthrough',
#     verbose_feature_names_out=False
# )
#
#
# preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('email_attempts_5m', ...), ('email_attempts_1h', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transfo

In [6]:
def add_device_attempts_5m_inplace(
    df: pd.DataFrame,
    *,
    device_col: str = "device_mac_id",
    time_col: str = "time_to_attempt",
    feature_col: str = "device_attempts_5m",
    window_minutes: int = 5,
) -> pd.DataFrame:
    """
    Add the number of previous attempts made by the same device
    during the previous five minutes.

    The current attempt is excluded.

    This function modifies the supplied DataFrame by adding one column.
    It does not reorder the DataFrame.
    """

    # ---------------------------------------------------------
    # Step 1: Validate required columns
    # ---------------------------------------------------------

    required_columns = {
        device_col,
        time_col,
    }

    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    if df[device_col].isna().any():
        raise ValueError(
            f"Column '{device_col}' contains missing device values."
        )

    # ---------------------------------------------------------
    # Step 2: Convert timestamps
    # ---------------------------------------------------------

    parsed_time = pd.to_datetime(
        df[time_col],
        utc=True,
        errors="raise",
    )

    if parsed_time.isna().any():
        raise ValueError(
            f"Column '{time_col}' contains missing timestamps."
        )

    number_of_rows = len(df)

    if number_of_rows == 0:
        df[feature_col] = pd.Series(dtype="int64")
        return df

    # Convert timestamps to integer nanoseconds.
    #
    # Example:
    # 2026-07-01 10:00:00 becomes one integer value.
    #
    # Integer comparison is cheaper than repeatedly comparing
    # pandas Timestamp objects.
    timestamps_ns = parsed_time.array.asi8

    # ---------------------------------------------------------
    # Step 3: Convert device values to integer codes
    # ---------------------------------------------------------

    # Example:
    #
    # Device-A -> 0
    # Device-B -> 1
    # Device-C -> 2
    #
    # This avoids repeatedly using long device strings as
    # dictionary keys.
    device_codes, unique_devices = pd.factorize(
        df[device_col],
        sort=False,
    )

    # ---------------------------------------------------------
    # Step 4: Sort only row positions
    # ---------------------------------------------------------

    # Suppose original positions are:
    #
    # [0, 1, 2, 3]
    #
    # After sorting by timestamp, the order might become:
    #
    # [2, 0, 3, 1]
    #
    # The DataFrame itself is not sorted or copied.
    chronological_positions = np.argsort(
        timestamps_ns,
        kind="stable",
    )

    # Five minutes expressed in nanoseconds.
    window_ns = pd.Timedelta(
        minutes=window_minutes
    ).value

    # ---------------------------------------------------------
    # Step 5: Prepare state
    # ---------------------------------------------------------

    # device code -> recent timestamps
    #
    # Example:
    #
    # {
    #     0: deque([10:00, 10:02]),
    #     1: deque([10:01])
    # }
    recent_attempts_by_device = defaultdict(deque)

    # The output array follows original DataFrame row positions.
    counts = np.zeros(
        number_of_rows,
        dtype=np.int64,
    )

    # ---------------------------------------------------------
    # Step 6: Process rows chronologically
    # ---------------------------------------------------------

    for row_position in chronological_positions:

        current_time_ns = timestamps_ns[row_position]
        current_device_code = device_codes[row_position]

        cutoff_time_ns = (
            current_time_ns - window_ns
        )

        recent_timestamps = (
            recent_attempts_by_device[current_device_code]
        )

        # Remove timestamps that are more than five minutes old.
        #
        # Each timestamp enters the deque once and leaves once.
        while (
            recent_timestamps
            and recent_timestamps[0] < cutoff_time_ns
        ):
            recent_timestamps.popleft()

        # Everything currently remaining belongs to:
        #
        # [current time - 5 minutes, current time]
        #
        # The current event is not yet inside the deque.
        counts[row_position] = len(recent_timestamps)

        # Add the current event after calculating its feature.
        recent_timestamps.append(current_time_ns)

    # ---------------------------------------------------------
    # Step 7: Add the feature to the original DataFrame
    # ---------------------------------------------------------

    # counts already follows the original row positions.
    # No reverse sorting or merge is necessary.
    df[feature_col] = counts

    return df

In [10]:
pd.set_option('display.max_rows', 100)
print(df.columns)
add_device_attempts_5m_inplace(df)

df[
    [
        "id",
        "device_mac_id",
        "time_to_attempt",
        "device_attempts_5m",
    ]
].loc[df['email']=='user4@example.com'].head(100)

Index(['id', 'email', 'device_mac_id', 'ip', 'location', 'time_to_attempt',
       'rejection_reason', 'is_suspicious', 'scenario', 'device_attempts_5m'],
      dtype='object')


,id,device_mac_id,time_to_attempt,device_attempts_5m
9,10,02:E5:8E:03:51:D8,2026-07-01 02:33:34+00:00,0
10,11,02:E5:8E:03:51:D8,2026-07-01 02:36:11+00:00,1
14,15,02:E5:8E:03:51:D8,2026-07-01 04:02:48+00:00,0
17,18,02:E5:8E:03:51:D8,2026-07-01 04:52:14+00:00,0
32,33,02:E5:8E:03:51:D8,2026-07-01 13:09:44+00:00,0
62,63,02:E5:8E:03:51:D8,2026-07-02 00:42:55+00:00,0
72,73,02:E5:8E:03:51:D8,2026-07-02 07:14:31+00:00,0
78,79,02:E5:8E:03:51:D8,2026-07-02 08:50:27+00:00,0
101,102,02:E5:8E:03:51:D8,2026-07-02 15:39:41+00:00,0
141,142,02:E5:8E:03:51:D8,2026-07-03 00:49:26+00:00,0


In [5]:
addition_features=['device_attempts_5m',
                   'device_attempts_1h',
                   'ip_unique_devices_1h',
                   'device_unique_ips_1h',
                   'email_ip_attempts_5m',
                   'email_device_attempts_1h',
                   'is_new_email_for_ip',
                   'is_new_email_for_device',

                   'email_ip_switches_1h',
                   'email_device_switches_24h',
                   'email_location_changes_24h',
                   'ip_email_spread_ratio_1h']


 

Index(['id', 'email', 'device_mac_id', 'ip', 'location', 'time_to_attempt',
       'rejection_reason', 'is_suspicious', 'scenario'],
      dtype='object')

In [8]:
# final_df=preprocessor.fit_transform(df)
# final_df.info()

a=final_df.loc[final_df["email"] == "user4@example.com"] 
pd.set_option("display.max_rows", 100)

a.head(100)


,email_attempts_5m,email_attempts_1h,email_attempts_24h,ip_attempts_5m,ip_attempts_1h,email,time_to_attempt,ip,id,device_mac_id,location,rejection_reason,is_suspicious,scenario
9,0,0,0,0,0,user4@example.com,2026-07-01 02:33:34+00:00,192.0.2.88,10,02:E5:8E:03:51:D8,"Rajshahi, BD",expired_session,0,normal_user_error
10,0,0,0,0,0,user4@example.com,2026-07-01 02:36:11+00:00,192.0.2.88,11,02:E5:8E:03:51:D8,"Rajshahi, BD",password_typo,0,normal_user_error
14,1,1,0,1,1,user4@example.com,2026-07-01 04:02:48+00:00,192.0.2.88,15,02:E5:8E:03:51:D8,"Rajshahi, BD",unknown_device,0,normal_user_error
17,1,0,0,1,0,user4@example.com,2026-07-01 04:52:14+00:00,192.0.2.88,18,02:E5:8E:03:51:D8,"Rajshahi, BD",password_typo,0,normal_user_error
32,1,1,0,1,1,user4@example.com,2026-07-01 13:09:44+00:00,192.0.2.88,33,02:E5:8E:03:51:D8,"Rajshahi, BD",password_typo,0,normal_user_error
62,1,1,0,1,1,user4@example.com,2026-07-02 00:42:55+00:00,192.0.2.88,63,02:E5:8E:03:51:D8,"Rajshahi, BD",otp_failed,0,normal_user_error
72,1,1,0,1,1,user4@example.com,2026-07-02 07:14:31+00:00,192.0.2.88,73,02:E5:8E:03:51:D8,"Rajshahi, BD",password_typo,0,normal_user_error
78,1,1,0,1,1,user4@example.com,2026-07-02 08:50:27+00:00,192.0.2.88,79,02:E5:8E:03:51:D8,"Rajshahi, BD",invalid_password,0,normal_user_error
101,1,1,0,1,1,user4@example.com,2026-07-02 15:39:41+00:00,192.0.2.88,102,02:E5:8E:03:51:D8,"Rajshahi, BD",invalid_password,0,normal_user_error
141,1,1,0,1,1,user4@example.com,2026-07-03 00:49:26+00:00,192.0.2.88,142,02:E5:8E:03:51:D8,"Rajshahi, BD",invalid_password,0,normal_user_error
